In [5]:
import os 
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GOOGLE_API_KEY")
huggingface_api_key = os.getenv("HUGGINGFACE_API_KEY")
os.environ["HF_TOKEN"] = huggingface_api_key
# gemini_api_key

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Institutional Support page 27.pdf")
pages = loader.load_and_split()
# pages[0].page_content

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunked_content = splitter.split_documents(pages)
# chunked_content

In [13]:
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 802.81it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
from langchain_chroma import Chroma
vectorstoredb = Chroma.from_documents(
    documents=chunked_content,
    embedding=embedding
)
retriever = vectorstoredb.as_retriever(search_type="similarity", search_kwargs={"k": 5})
# retriever

In [15]:
respond = retriever.invoke("What are the characteristics of an entrepreneur?")

In [16]:
respond

[Document(id='baa81d82-ea74-4ea8-8423-bf399cd3f736', metadata={'moddate': '2025-12-15T09:59:41+05:30', 'total_pages': 88, 'producer': '2.2.15 (4.2.4)', 'source': 'Institutional Support page 27.pdf', 'creationdate': '2021-06-01T12:39:20+05:30', 'author': 'Lenovo-pc', 'page': 4, 'page_label': '5', 'creator': 'Microsoft® Word 2013'}, page_content='business enterprise. Joseph A Schumpeter defines an entrepreneur as “ one who innovates, \nraises money, assembles inputs and sets the organization going with th e ability to identify \nthem and opportunities, which others are not able to fulfil such economic opportunities”. He \nfurther said, “An entrepreneur is an innovator playing the role of a dynamic businessman \nadding material growth to economic development”. \nCharacteristics of Entrepreneur   \nAn entrepreneur is a highly achievement oriented, enthusiastic and energetic individual. He \nis a business leader. He has the following characteristic: \n1) An entrepreneur brings about change 

In [17]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise accordingly the dictionary below given.

Question: {question}
Context: {context}
Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

In [18]:
from langchain_google_genai import GoogleGenerativeAI
llm = GoogleGenerativeAI(model="models/gemini-2.5-flash")

In [36]:
llm.invoke("Hi I am manas, how are you?")

'Hello Manas! It\'s nice to meet you.\n\nAs an AI, I don\'t have feelings or a physical state like humans do, so I don\'t "feel" good or bad. However, I\'m operating perfectly and ready to assist you!\n\nHow can I help you today?'

In [19]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

rag_chain = (
    {"context":retriever , "question":RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [20]:
res = rag_chain.invoke("What are Characteristics of Entrepreneur?")

In [ ]:
import textwrap

print(textwrap.fill(res, width=80))

An entrepreneur is a highly achievement-oriented, enthusiastic, and energetic
individual who acts as a business leader and a catalyst for change. They are
action-oriented, highly motivated, and willing to take risks to achieve goals.
Entrepreneurs accept responsibilities with enthusiasm and endurance, functioning
as both thinkers and doers, planners and workers. They can foresee the future,
seize market opportunities with persuasiveness, and skillfully manipulate funds.
An entrepreneur also possesses the precision to detect errors and deficiencies.
Ventures are undertaken not just for personal gain, but also for the benefit of
consumers, government, and society. They build new enterprises, demonstrating
intense determination to overcome hurdles and complete jobs. Joseph A.
Schumpeter defines an entrepreneur as one who innovates, raises money, assembles
inputs, and identifies economic opportunities. They play a dynamic role in
adding material growth to economic development.
